In [0]:
from pyspark.sql import functions as F

new_rows = (
    spark.table("samples.tpch.orders")
    .filter("o_orderdate = '1997-12-30'")
    .withColumn("ingested_at", F.current_timestamp())   # a column the table doesn't have
)

new_rows.write.mode("append").saveAsTable("workspace.bronze.orders_1998")

[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 0cffd9b6-c85c-439c-b4b5-7c9c2d982b6d).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.


Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TABLE command for changing the schema.

Schema merge is not true, either merge is to be enabled ot need to alter the table to let this run.


In [0]:
from pyspark.sql import functions as F

new_rows = (
    spark.table("samples.tpch.orders")
    .filter("o_orderdate = '1997-12-30'")
    .withColumn("ingested_at", F.current_timestamp())   # a column the table doesn't have
)

(
    new_rows.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.bronze.orders_1998")
)

In [0]:
%sql
SELECT * FROM workspace.bronze.orders_1998 
WHERE ingested_at is NOT NULL
LIMIT 10

In [0]:
from pyspark.sql import functions as F

def table_summary(table_name: str) -> dict:
    """Return a quick health summary for a Delta table."""
    row_count = spark.table(table_name).count()
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    latest = history.orderBy(F.desc("version")).first()

    return {
        "table": table_name,
        "rows": row_count,
        "latest_version": latest["version"],
        "last_operation": latest["operation"],
        "last_changed": latest["timestamp"],
        "total_versions": history.count()   # hint: history.count()
    }

tables = [
    "workspace.bronze.orders_1998",
    "workspace.bronze.does_not_exist",
]

for t in tables:
    try:
        print(table_summary(t))
    except Exception as e:
        print({"table": t, "error": str(e)})

In [0]:
# 1. Create a volume: a governed folder for files (not tables)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.landing")
base = "/Volumes/workspace/bronze/landing"

df = spark.table("samples.tpch.orders").filter("o_orderdate >= '1998-07-01'")

# 2. Write the SAME data twice: plain Parquet, and Delta
df.write.mode("overwrite").parquet(f"{base}/orders_parquet")
df.write.mode("overwrite").format("delta").save(f"{base}/orders_delta")

# 3. Compare what's on disk
display(dbutils.fs.ls(f"{base}/orders_parquet"))
display(dbutils.fs.ls(f"{base}/orders_delta"))
display(dbutils.fs.ls(f"{base}/orders_delta/_delta_log"))

In [0]:
%sql
-- 4. Try to UPDATE each one
UPDATE delta.`/Volumes/workspace/bronze/landing/orders_delta`
SET o_orderstatus = 'X' WHERE o_orderkey < 1000;



In [0]:
%sql
UPDATE parquet.`/Volumes/workspace/bronze/landing/orders_parquet`
SET o_orderstatus = 'X' WHERE o_orderkey < 1000;     -- expect this to fail

In [0]:
base = "/Volumes/workspace/bronze/landing"
log = spark.read.json(f"{base}/orders_delta/_delta_log/*.json")

display(log)

What’s in the Delta folder that isn’t in the Parquet folder?

_delta_log/

Which UPDATE failed, and what was the error?

Update to parquet folder file failed with the error "[UC_COMMAND_NOT_SUPPORTED.WITHOUT_RECOMMENDATION] The command(s): UpdateTable are not supported in Unity Catalog.  SQLSTATE: 0AKUC"

In the log output, which columns show the add and remove actions? What did your UPDATE do to the files?

There are add and remove column in the json log files, but I am not sure where the UPDATE related info go.



In [0]:
from pyspark.sql import functions as F
base = "/Volumes/workspace/bronze/landing"
log = (
    spark.read.json(f"{base}/orders_delta/_delta_log/*.json")
    .select("*", F.col("_metadata.file_name").alias("log_file"))
)

display(
    log.select(
        "log_file",
        F.col("commitInfo.operation").alias("operation"),
        F.col("add.path").alias("file_added"),
        F.col("remove.path").alias("file_removed"),
    ).orderBy("log_file")
)